In [ ]:
import os
repo_url = "https://github.com/aaronjji/rk8391-devbox.git"
if not os.path.exists("/kaggle/working/gave2-challenge"):
    os.system(f"git clone --recurse-submodules {repo_url} /kaggle/working/gave2-challenge")
os.chdir("/kaggle/working/gave2-challenge")
os.system("git log -1 --oneline")
print(os.getcwd())

In [ ]:
os.system("pip install -q albumentations opencv-python-headless scikit-image scipy networkx sknw huggingface_hub safetensors")

In [ ]:
candidates = [
    "/kaggle/input/datasets/aaronajit/gave2-preliminary/GAVE2_preliminary",
    "/kaggle/input/gave2-preliminary/GAVE2_preliminary",
]
os.system("find /kaggle/input -maxdepth 3")
DATA_ROOT = None
for c in candidates:
    if os.path.exists(f"{c}/validation/images"):
        DATA_ROOT = c
        break
assert DATA_ROOT is not None, f"Dataset not found in {candidates}"
print("DATA_ROOT =", DATA_ROOT)

reg_candidates = [
    "/kaggle/input/datasets/aaronajit/gave2-registered/registered",
    "/kaggle/input/gave2-registered/registered",
]
FFA_ROOT = None
for c in reg_candidates:
    if os.path.exists(f"{c}/validation/FFA_A"):
        FFA_ROOT = c
        break
assert FFA_ROOT is not None, f"Registered FFA dataset not found in {reg_candidates}"
print("FFA_ROOT =", FFA_ROOT)

def find_ckpt_dir(name):
    for c in [f"/kaggle/input/datasets/aaronajit/{name}", f"/kaggle/input/{name}"]:
        if os.path.exists(c):
            return c
    raise AssertionError(f"{name} checkpoint dir not found")

TASK1_CKPT_DIR = find_ckpt_dir("gave2-task1-7fold-ckpts")
TASK2_CKPT_DIR = find_ckpt_dir("gave2-task2-7fold-ckpts")
print("TASK1_CKPT_DIR =", TASK1_CKPT_DIR)
print("TASK2_CKPT_DIR =", TASK2_CKPT_DIR)

TASK1_CKPTS = " ".join(f"{TASK1_CKPT_DIR}/fold{i}_final.pth" for i in range(7))
TASK2_CKPTS = " ".join(f"{TASK2_CKPT_DIR}/fold{i}_final.pth" for i in range(7))
print("TASK1_CKPTS:", TASK1_CKPTS)
print("TASK2_CKPTS:", TASK2_CKPTS)

In [ ]:
# 7-fold ensemble inference, Task1 (3ch CFP), with horizontal-flip TTA --
# same recipe as the validated 5-fold+TTA submission, extended to 7 folds.
os.system(
    f"python -u src/predict_ensemble.py --task task1 --tta "
    f"--checkpoints {TASK1_CKPTS} "
    f"--images-dir {DATA_ROOT}/validation/images "
    f"--masks-dir {DATA_ROOT}/validation/masks "
    f"--out-dir predictions/task1/validation"
)

In [ ]:
# 7-fold ensemble inference, Task2 (5ch CFP+FFA), with horizontal-flip TTA
os.system(
    f"python -u src/predict_ensemble.py --task task2 --tta "
    f"--checkpoints {TASK2_CKPTS} "
    f"--images-dir {DATA_ROOT}/validation/images "
    f"--masks-dir {DATA_ROOT}/validation/masks "
    f"--ffa-dir {FFA_ROOT}/validation "
    f"--out-dir predictions/task2/validation"
)

In [ ]:
# Task3 biomarkers, Task1-sourced (validated as better than Task2-sourced)
os.system(
    f"python -u src/run_task3.py "
    f"--av-dir predictions/task1/validation "
    f"--images-dir {DATA_ROOT}/validation/images "
    f"--masks-dir {DATA_ROOT}/validation/masks "
    f"--out-dir predictions/task3/validation"
)

In [ ]:
# Package into the official submission zip structure
os.system(
    "python -u src/format_submission.py --team-id aaronteam "
    "--task1-dir predictions/task1/validation "
    "--task2-dir predictions/task2/validation "
    "--task3-dir predictions/task3/validation "
    "--out-zip submissions/aaronteam_7fold-ensemble-tta.zip"
)
os.system("ls -la submissions/")